<a href="https://colab.research.google.com/github/npincini/BUSA3022/blob/main/BUSA3022_Data_Pipeline_(S%26P_ASX_200_%2B_VIX).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BUSA3022 Data Pipeline (S&P/ASX 200 + VIX)

This notebook is the proof of concept for a data pipeline. It takes two public sources from the internet and transforms them into one curated CSV.

| Source | What it is | Role in the pipeline |
|---|---|---|
| Yahoo Finance via `yfinance` (`^AXJO`) | S&P/ASX 200 daily open, high, low, close, volume | **Market index series** (Performance of Australian Market) |
| FRED (`VIXCLS`) | CBOE Volatility Index (VIX) daily close | **Macro series** (US market's expectation of volatility) |


**Design rules this notebook follows**

1. **Storage is separate from transformation.** Ingestion saves an untouched, timestamped copy of each download into `data/raw/`. The transform step reads from those files on disk.
2. **Nothing is silently dropped.** Any row that fails a check goes to a quarantine file with a reason code. For each source, `raw rows = clean rows + quarantined rows`, and the notebook stops if that does not balance.
3. **Every step is logged** (what happened, when, how many rows, and whether it worked).
4. **Fail loudly.** A changed file layout (such as a schema change) stops the run with a clear error rather than producing a wrong file.

**How to run in Colab:** `Runtime > Run all`. Outputs appear in `data/` and `logs/`.

## Step 0. Setup

`yfinance` is an unofficial library that breaks when Yahoo changes its site, so we upgrade it first. If you see a `yfinance` error later, restart the runtime and run again after this install.

In [1]:
%pip install -q --upgrade yfinance # yfinance is a python module that allows the download of finance data from Yahoo

In [2]:
# --- Python standard library (built into Python, nothing to install) ---
import csv              # write the log files as CSV
import hashlib          # compute file "fingerprints" (SHA-256) for the raw-file manifest
import json             # store a rejected row's original contents as text in the quarantine file
import logging          # print log messages and also write them to a .log file
import time             # pause between download retries
import urllib.parse     # build the FRED web address (URL)
import urllib.request   # download the FRED CSV
from pathlib import Path  # work with file and folder paths

# --- Third-party libraries (pre-installed in Colab; yfinance is upgraded in Step 0) ---
import numpy as np      # number and array helpers
import pandas as pd     # tables ("DataFrames"): the main tool for cleaning and joining data
import yfinance as yf   # downloads Yahoo Finance data

print("pandas", pd.__version__, "| yfinance", yf.__version__)

pandas 2.2.3 | yfinance 1.7.0


### Configuration

Every setting is in this one cell, so the pipeline can be tuned (or handed to someone else) without hunting through code.

* **History window:** 2 years of ASX data. This gives a long history should someone like to utilise metrics that involve long periods of time.
* **VIX look-back:** Fetch a little extra VIX history before the ASX window starts, so the earliest ASX dates still have a previous VIX close to pair with.
* **Staleness limit:** A VIX value is never carried forward more than 7 days. Beyond that the VIX cell is left blank and logged rather than quietly reusing an old number. This ensures that old data is not utilised.

In [3]:
# ALL SETTINGS ARE IN THIS CELL. Change a value here and every later step uses it.
# Nothing is downloaded or changed in this cell; it only sets up names and numbers.

# --- Sources -------------------------------------------------------------
ASX_TICKER = "^AXJO"                     # S&P/ASX 200 on Yahoo Finance
FRED_SERIES = "VIXCLS"                   # VIX daily close on FRED
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"   # no API key needed

# --- Window and time zone ------------------------------------------------
LOCAL_TZ = "Australia/Sydney"
HISTORY_YEARS = 2
VIX_LOOKBACK_DAYS = 14                   # extra VIX history before the ASX window
VIX_MAX_STALENESS_DAYS = 7               # never carry a VIX close forward further than this

# --- Validation thresholds -----------------------------------------------
ASX_CLOSE_CUTOFF = "16:30"               # before this (Sydney time) today's ASX bar may be incomplete
OHLC_TOLERANCE = 0.0001                  # 0.01% leeway when checking high >= open/close >= low
MAX_CALENDAR_GAP_DAYS = 5                # warn if consecutive ASX rows are more than 5 days apart
FRESHNESS_LIMIT_DAYS = 5                 # warn if the newest row is older than this
LOW_VOLUME_FRACTION = 0.2                # warn on days whose volume is under 20% of the median volume
OPEN_MATCH_WARN_SHARE = 0.5              # warn if the 'open' equals the previous close on more than this share of days

# --- Network behaviour ---------------------------------------------------
RETRY_ATTEMPTS = 3
RETRY_BASE_DELAY_S = 2                   # waits 2s, then 4s between attempts
HTTP_TIMEOUT_S = 60                      # seconds to wait for FRED to respond

# --- Run clock (always Sydney time, regardless of where Colab runs) -------
NOW_LOCAL = pd.Timestamp.now(tz=LOCAL_TZ)                # the current moment, in Sydney time
RUN_ID = NOW_LOCAL.strftime("%Y%m%dT%H%M%S")             # unique label for this run, e.g. 20261026T132501; used in every file name
TODAY = NOW_LOCAL.tz_localize(None).normalize()          # today's Sydney date (no time, no zone)
START_DATE = TODAY - pd.DateOffset(years=HISTORY_YEARS)  # first date of ASX history we ask for
VIX_START = START_DATE - pd.Timedelta(days=VIX_LOOKBACK_DAYS)  # VIX starts a bit earlier so the first ASX dates still have a previous VIX close

print(f"Run ID: {RUN_ID}")
print(f"ASX window: {START_DATE:%Y-%m-%d} to {TODAY:%Y-%m-%d}  |  VIX fetched from {VIX_START:%Y-%m-%d}")

Run ID: 20261026T135317
ASX window: 2024-10-26 to 2026-10-26  |  VIX fetched from 2024-10-12


### Folders and logging

Three folders are used in this pipeline:

* `data/raw/` holds untouched downloads plus a `manifest.csv` (what was downloaded, when, how many rows, and a file fingerprint).
* `data/curated/` holds the final transformed file.
* `logs/` holds three records of every run:
  * `quality_log_<run>.csv`: every event, with a level and a code.
  * `quarantine_<run>.csv`: every rejected row, with the reason it was rejected and its original contents.
  * `run_<run>.log`: the same events as readable text.

Log files are written as the run progresses, so even a run that crashes leaves evidence of how far it got.

In [4]:
# ---- Folders: raw = untouched downloads, curated = final output, logs = what happened ----
ROOT = Path(".")
RAW_DIR = ROOT / "data" / "raw"
CURATED_DIR = ROOT / "data" / "curated"
LOG_DIR = ROOT / "logs"
for d in (RAW_DIR, CURATED_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- File names. Each run gets its own log files, named with the RUN_ID ----
EVENT_LOG = LOG_DIR / f"quality_log_{RUN_ID}.csv"
QUARANTINE_LOG = LOG_DIR / f"quarantine_{RUN_ID}.csv"
RUN_LOG = LOG_DIR / f"run_{RUN_ID}.log"
MANIFEST = RAW_DIR / "manifest.csv"
CURATED_FILE = CURATED_DIR / "market_curated.csv"

# ---- Column names for the three CSV files the pipeline writes as it goes ----
EVENT_FIELDS = ["run_id", "timestamp", "step", "level", "code", "message", "count"]
QUARANTINE_FIELDS = ["run_id", "source", "raw_row", "reason_code", "detail", "raw_record"]
MANIFEST_FIELDS = ["run_id", "source", "origin", "file", "rows", "sha256", "fetched_at", "status"]


def init_csv(path, fields):
    """Create a CSV file containing only a header row, unless the file already exists.

    Used so the log files exist from the very start of a run. Even a run with no problems
    leaves an empty quarantine file, which is evidence that nothing was rejected.
    """
    if not path.exists():
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(fields)


# ---- Create the (empty) log files now ----
init_csv(EVENT_LOG, EVENT_FIELDS)
init_csv(QUARANTINE_LOG, QUARANTINE_FIELDS)   # created even if nothing is rejected, so "zero rejects" is auditable
init_csv(MANIFEST, MANIFEST_FIELDS)

# ---- Logging setup: every message goes to the screen AND to a .log file, with Sydney timestamps ----
logger = logging.getLogger("pipeline")
logger.setLevel(logging.INFO)
logger.propagate = False                       # Colab's root logger would otherwise print every line twice
logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s %(levelname)-8s %(message)s")
_fmt.converter = lambda ts: pd.Timestamp(ts, unit="s", tz="UTC").tz_convert(LOCAL_TZ).timetuple()   # log in Sydney time
for _h in (logging.FileHandler(RUN_LOG, encoding="utf-8"), logging.StreamHandler()):
    _h.setFormatter(_fmt)
    logger.addHandler(_h)

# ---- Counters shared by later steps ----
EVENTS = []                                    # in-memory copy of the event log, used for the run summary
QUARANTINED = {"asx": 0, "vix": 0}             # rows rejected per source, used for reconciliation
COUNTS = {}                                    # raw / clean / quarantined row counts per source


# ---- Custom error types, so a failure says exactly what went wrong ----
class SchemaError(Exception):
    """Raised when a source file does not have the columns we expect (the source changed its layout)."""


class PipelineError(Exception):
    """Raised when an integrity check fails (for example the row counts do not add up)."""


def log_event(step, level, code, message, count=None):
    """Record one thing that happened. Every step of the pipeline reports through this function.

    One call does three things:
      1. adds a row to logs/quality_log_<run>.csv (the audit record),
      2. prints the message and writes it to logs/run_<run>.log (the readable record),
      3. keeps a copy in the EVENTS list, used for the summary tables at the end.

    step    which stage reported it, e.g. "ingest", "validate_asx", "align"
    level   how serious: INFO (normal), WARNING (look at this), ERROR (a fallback was used), CRITICAL (run stopped)
    code    short fixed label for the event, e.g. "FETCH_OK" or "ZERO_RANGE_BAR"; easy to filter on
    message a readable explanation
    count   optional number of rows affected
    """
    row = {
        "run_id": RUN_ID,
        "timestamp": pd.Timestamp.now(tz=LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S%z"),
        "step": step, "level": level, "code": code, "message": message, "count": count,
    }
    EVENTS.append(row)
    with open(EVENT_LOG, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=EVENT_FIELDS).writerow(row)
    suffix = f" (n={count})" if count is not None else ""
    logger.log(getattr(logging, level), "[%s] %s: %s%s", step, code, message, suffix)


# First entry in the log: records the library versions and date window used for this run.
log_event("setup", "INFO", "RUN_START", f"yfinance {yf.__version__}, pandas {pd.__version__}; window {START_DATE:%Y-%m-%d} to {TODAY:%Y-%m-%d}")

2026-10-26 13:53:17,835 INFO     [setup] RUN_START: yfinance 1.7.0, pandas 2.2.3; window 2024-10-26 to 2026-10-26


## Step 1. Ingest (raw layer)

Ingestion has one job: get the data and save it exactly as received, with no cleaning. This is what separates storage from transformation. If a cleaning rule is later found to be wrong, we can rerun it on the saved raw file without re-downloading.

**Repeatable:** each run writes new timestamped files (`asx200_yahoo_<run>.csv`, `vix_fred_<run>.csv`) and adds a line to `manifest.csv`. The same notebook can be scheduled (for example a daily GitHub Actions job or `cron` running the exported script) and would build a history of raw snapshots.

**Failure handling (network):** each download is attempted up to three times with a growing wait between attempts, and every failed attempt is logged. If all attempts fail, the pipeline falls back to the most recent raw file available and logs this as an `ERROR`. If there is no earlier file, it stops with a `CRITICAL` event. The data's age is then checked in Step 2, so a stale fallback cannot pass unnoticed.

**If FRED stays unreachable:** download the CSV from the FRED page for `VIXCLS` (`Download > CSV`) and save it in `data/raw/` as `vix_fred_<YYYYMMDDTHHMMSS>_manual.csv`, using a timestamp later than any existing raw file. The fallback then treats it as the latest raw file and it is validated like any other download.

**Time zones at the source:** Yahoo returns ASX dates as midnight Sydney time (with an offset like `+11:00`). FRED returns plain US calendar dates. These are handled explicitly in Step 2.

In [5]:
# STEP 1 HELPER FUNCTIONS (INGEST). This cell only DEFINES functions; the next cell runs them.
# Ingestion's job: download each source and save it exactly as received. No cleaning happens here.


def with_retries(fn, what):
    """Run a download function; if it fails, wait and try again, up to RETRY_ATTEMPTS times.

    Waits 2 seconds after the first failure, 4 after the second. Every failed attempt is logged. If every attempt fails, the last error is raised again so the caller can decide what to do (ingest_asx and ingest_vix then fall back to the newest older raw file).

    fn    the function to run, e.g. fetch_vix (called with no arguments)
    what  a label used in log messages, e.g. "FRED VIXCLS"
    """
    last_error = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            return fn()
        except Exception as exc:  # network errors, empty responses, library errors
            last_error = exc
            log_event("ingest", "WARNING", "FETCH_ATTEMPT_FAILED",
                      f"{what}: attempt {attempt}/{RETRY_ATTEMPTS}: {type(exc).__name__}: {exc}")
            if attempt < RETRY_ATTEMPTS:
                time.sleep(RETRY_BASE_DELAY_S * 2 ** (attempt - 1))
    raise last_error


def sha256_file(path):
    """Return a file's SHA-256 "fingerprint": a long code that only matches if the file is byte-for-byte identical.

    It is stored in the manifest, so anyone can later confirm a raw file has not been edited.
    """
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def register_raw(source, origin, path, rows, status):
    """Add one line to data/raw/manifest.csv recording what was saved in the raw layer.

    The line holds: the run, which source, where it came from, the file name, the row count, the file fingerprint, the time, and a status ("ok" = fresh download, "fallback" = an older file was reused).
    """
    row = {
        "run_id": RUN_ID, "source": source, "origin": origin, "file": Path(path).name,
        "rows": rows, "sha256": sha256_file(path),
        "fetched_at": pd.Timestamp.now(tz=LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S%z"), "status": status,
    }
    with open(MANIFEST, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=MANIFEST_FIELDS).writerow(row)


def latest_raw(prefix):
    """Find the newest raw file for a source, e.g. latest_raw("vix_fred"). Returns None if there are none yet.

    Raw file names end with the run timestamp (20261026T132501), so sorting the names alphabetically also sorts them by time, and the last one is the newest. Used in two places: as the fallback when a download fails, and as the default input to the transform step.
    """
    files = sorted(RAW_DIR.glob(f"{prefix}_*.csv"))
    return files[-1] if files else None


# ---- ASX 200 from Yahoo Finance ------------------------------------------
def fetch_asx():
    """Download about two years of daily S&P/ASX 200 data from Yahoo Finance and return it as a table.

    This only DOWNLOADS. It does not save or clean anything (ingest_asx does the saving). It raises an error if Yahoo returns nothing, so with_retries will try again.
    """
    df = yf.Ticker(ASX_TICKER).history(
        start=f"{START_DATE:%Y-%m-%d}",
        end=f"{TODAY + pd.Timedelta(days=1):%Y-%m-%d}",   # end date is exclusive
        auto_adjust=False,                                 # keep raw prices, not dividend-adjusted ones
        actions=False,
    )
    if df is None or df.empty:
        raise ValueError("Yahoo Finance returned no rows")
    if isinstance(df.columns, pd.MultiIndex):              # some yfinance versions return two-level headers
        log_event("ingest", "WARNING", "SCHEMA_VARIATION", "Yahoo returned multi-level column headers; flattened")
        df.columns = df.columns.get_level_values(0)
    return df


def ingest_asx():
    """Complete ingestion step for the ASX 200. Returns the path of the raw file the next step should read.

    1. Download (with retries) using fetch_asx.
    2. Save the result, untouched, in data/raw/ and record it in the manifest.
    3. If the download fails after every retry: reuse the newest older raw file (logged as ERROR), or stop with a CRITICAL log entry if there is no older file.
    """
    path = RAW_DIR / f"asx200_yahoo_{RUN_ID}.csv"
    try:
        df = with_retries(fetch_asx, f"Yahoo Finance {ASX_TICKER}")
        df.reset_index().to_csv(path, index=False)         # the date index becomes the first column
        register_raw("asx", f"yfinance:{ASX_TICKER}", path, len(df), "ok")
        log_event("ingest", "INFO", "FETCH_OK", f"Saved raw ASX file {path.name}", count=len(df))
        return path
    except Exception as exc:
        fallback = latest_raw("asx200_yahoo")
        if fallback is None:
            log_event("ingest", "CRITICAL", "FETCH_FAILED_NO_FALLBACK", f"ASX download failed and no earlier raw file exists: {exc}")
            raise
        log_event("ingest", "ERROR", "FALLBACK_TO_LAST_RAW", f"ASX download failed ({exc}); using {fallback.name}")
        register_raw("asx", f"fallback:{fallback.name}", fallback, "", "fallback")
        return fallback


# ---- VIX from FRED ---------------------------------------------------------
def fetch_vix():
    """Download the VIX history from FRED and return it as raw bytes (the file's exact contents, not a table).

    Builds the web address from the series name (VIXCLS) and the date window, then downloads it with Python's built-in urllib. That is the same method pandas.read_csv(url) uses. Before returning, it checks that the first line mentions "VIXCLS". If the server sent an error web page instead of a CSV, this check fails, so a bad page is never saved as if it were data.
    """
    params = {"id": FRED_SERIES, "cosd": f"{VIX_START:%Y-%m-%d}", "coed": f"{TODAY:%Y-%m-%d}"}
    url = f"{FRED_URL}?{urllib.parse.urlencode(params)}"
    with urllib.request.urlopen(url, timeout=HTTP_TIMEOUT_S) as response:
        content = response.read()
    first_line = content.decode("utf-8", "replace").splitlines()[0]
    if FRED_SERIES not in first_line:                      # e.g. an HTML error page instead of a CSV
        raise ValueError(f"Unexpected response from FRED; first line was: {first_line[:80]!r}")
    return content


def ingest_vix():
    """Complete ingestion step for the VIX. Returns the path of the raw file the next step should read.

    Same pattern as ingest_asx: download with retries, save the file exactly as received (no parsing), record it in the manifest, and fall back to the newest older raw file if the download fails.
    """
    path = RAW_DIR / f"vix_fred_{RUN_ID}.csv"
    try:
        content = with_retries(fetch_vix, f"FRED {FRED_SERIES}")
        path.write_bytes(content)                          # saved byte-for-byte as received
        rows = len(content.decode("utf-8", "replace").splitlines()) - 1
        register_raw("vix", f"fred:{FRED_SERIES}", path, rows, "ok")
        log_event("ingest", "INFO", "FETCH_OK", f"Saved raw VIX file {path.name}", count=rows)
        return path
    except Exception as exc:
        fallback = latest_raw("vix_fred")
        if fallback is None:
            log_event("ingest", "CRITICAL", "FETCH_FAILED_NO_FALLBACK", f"VIX download failed and no earlier raw file exists: {exc}")
            raise
        log_event("ingest", "ERROR", "FALLBACK_TO_LAST_RAW", f"VIX download failed ({exc}); using {fallback.name}")
        register_raw("vix", f"fallback:{fallback.name}", fallback, "", "fallback")
        return fallback

In [6]:
# RUN INGESTION. Each call downloads one source, saves it untouched in data/raw/, and returns the path of the saved file. Later steps read from these files, not from memory.
ASX_RAW_PATH = ingest_asx()
VIX_RAW_PATH = ingest_vix()
print("\nRaw files in use:", ASX_RAW_PATH.name, "|", VIX_RAW_PATH.name)

2026-10-26 13:53:18,234 INFO     [ingest] FETCH_OK: Saved raw ASX file asx200_yahoo_20261026T135317.csv (n=506)
2026-10-26 13:53:18,755 INFO     [ingest] FETCH_OK: Saved raw VIX file vix_fred_20261026T135317.csv (n=529)



Raw files in use: asx200_yahoo_20261026T135317.csv | vix_fred_20261026T135317.csv


## Step 2. Transform and validate

Both transform functions follow the same pattern: read the raw file, run checks in a fixed order, and send any failing row to quarantine. A row that fails one check is quarantined once (the first failure wins) and is not tested further.

### Row checks (failing rows are quarantined, with a reason code)

| Reason code | Applies to | Meaning |
|---|---|---|
| `INVALID_DATE` | both | The date could not be read |
| `DUPLICATE_DATE` | both | The date appears more than once (the last copy is kept) |
| `INCOMPLETE_SESSION` | ASX | Today's bar, fetched before the market has finished closing (a live partial bar) |
| `MISSING_VALUE` | both | A required value is blank, `.`, `NA`, `null`, etc. VIX rows with a blank value are logged at INFO level; if an ASX date would have used one, the align step uses the most recent earlier close |
| `UNPARSEABLE_VALUE` | both | A value is present but is not a number |
| `NON_POSITIVE_PRICE` | both | A price or VIX value is zero or negative |
| `OHLC_INCONSISTENT` | ASX | High is below the open, close or low (or low is above them), beyond a 0.01% tolerance |
| `NEGATIVE_VOLUME` | ASX | Volume is below zero |
| `ZERO_RANGE_BAR` | ASX | Open, high, low and close are all identical, so the bar has zero trading range |

### Dataset checks (rows are kept, but a warning is logged)

Repeated identical prices on consecutive trading days (stale feed), gaps of more than 5 calendar days (possible missing days), days with far lower volume than normal (or volume that is mostly zero), an `open` that just repeats the previous close, and data that is older than expected.

### Time zones in this step

* **ASX:** the raw timestamp (midnight Sydney, with an offset that changes between `+10:00` and `+11:00` for daylight saving) is converted explicitly to `Australia/Sydney` and reduced to a plain calendar date. This avoids off-by-one-day errors.
* **VIX:** FRED dates are already plain US market dates, so no conversion is needed. Alignment in Step 3 is what handles the time difference between the two markets.

**Yahoo data quality warnings:** Yahoo's `^AXJO` feed can report the previous day's close in place of the day's `open`, and can contain days with volume far below the median. The checks measure both on every run and log a warning when they apply.

In [7]:
# STEP 2 HELPER FUNCTIONS (VALIDATE). Small tools shared by the two transform functions below.
# This cell only DEFINES functions; nothing runs yet.

# Text values that mean "no data here" (FRED uses a blank or "." for a missing value).
MISSING_MARKERS = {"", ".", "na", "n/a", "nan", "null", "none"}


def parse_numeric(series):
    """Convert a column of text into numbers, and report which entries could not be converted and why.

    Returns three things, one value per row:
      numbers         the converted numbers (blank where conversion was not possible)
      is_missing      True where the text was a "no data" marker, e.g. "" or "."
      is_unparseable  True where there was something in the cell but it is not a number, e.g. "abc"
    Example: "12.5" -> 12.5   |   "." -> missing   |   "abc" -> unparseable
    Keeping "missing" and "not a number" separate lets the quarantine log give the correct reason.
    """
    text = series.astype(str).str.strip()
    is_missing = text.str.lower().isin(MISSING_MARKERS)
    numbers = pd.to_numeric(text.where(~is_missing), errors="coerce")
    is_unparseable = ~is_missing & numbers.isna()
    return numbers, is_missing, is_unparseable


def sample_dates(values, n=8):
    """Turn a list of dates into a short piece of text for log messages.

    Example: gives "2025-01-02, 2026-01-02". Shows at most n dates (earliest first) and adds " ..." if there are more, so a warning about 15 affected days stays on one readable line.
    """
    dates = pd.to_datetime(pd.Series(values)).dropna().sort_values()
    text = ", ".join(d.strftime("%Y-%m-%d") for d in dates.head(n))
    return text + (" ..." if len(dates) > n else "")


def quarantine(source, rows, code, detail, level="WARNING"):
    """Set rejected rows aside in the quarantine file instead of deleting them.

    For each rejected row it writes one line to logs/quarantine_<run>.csv containing: the source, the row's position in the raw file, the reason code, an explanation, and the row's ORIGINAL contents as text. So no rejected record is ever lost, and each one can be traced back to the raw file. It also adds the rows to the per-source rejected count (used by reconcile) and logs the event. Columns whose names start with "_" are helper columns added by the transform step, so they are left out.
    """
    if len(rows) == 0:
        return
    original_cols = [c for c in rows.columns if not c.startswith("_")]
    records = [
        {"run_id": RUN_ID, "source": source, "raw_row": int(raw_row), "reason_code": code,
         "detail": detail, "raw_record": json.dumps(rec, default=str)}
        for raw_row, rec in zip(rows["_raw_row"], rows[original_cols].to_dict("records"))
    ]
    pd.DataFrame(records, columns=QUARANTINE_FIELDS).to_csv(QUARANTINE_LOG, mode="a", header=False, index=False)
    QUARANTINED[source] += len(rows)
    log_event(f"validate_{source}", level, code, detail, count=len(rows))


def reject(source, df, mask, code, detail, level="WARNING"):
    """Apply ONE validation rule: quarantine the rows that break it and return only the rows that pass.

    mask is a True/False value for each row, where True means "this row breaks the rule". The transform functions call reject() many times in a row, one rule per call, so each call removes the rows that fail that rule. A row is removed by the first rule it breaks.
    """
    quarantine(source, df[mask], code, detail, level)
    return df[~mask]


def reconcile(source, n_raw, n_clean):
    """Prove that no rows vanished: rows read from the raw file must equal clean rows plus quarantined rows.

    This is the check behind the rule "bad records are never silently dropped". The result is logged, and if the numbers do not balance the run stops with an error.
    """
    q = QUARANTINED[source]
    balanced = n_raw == n_clean + q
    COUNTS[source] = {"raw_rows": n_raw, "clean_rows": n_clean, "quarantined_rows": q, "balanced": balanced}
    log_event(f"reconcile_{source}", "INFO" if balanced else "CRITICAL", "ROW_RECONCILIATION",
              f"raw={n_raw}, clean={n_clean}, quarantined={q}, balanced={balanced}")
    if not balanced:
        raise PipelineError(f"{source}: row counts do not reconcile (raw={n_raw}, clean={n_clean}, quarantined={q})")


def require_rows(source, clean):
    """Stop with a clear message if validation removed every row (for example the source sent an empty file)."""
    if len(clean) == 0:
        msg = f"No usable {source} rows remain after validation; check the raw file and the quarantine log"
        log_event(f"validate_{source}", "CRITICAL", "NO_USABLE_ROWS", msg)
        raise PipelineError(msg)


def check_freshness(source, latest):
    """Warn if the newest row is older than FRESHNESS_LIMIT_DAYS (e.g. because a failed download reused an old file)."""
    age = (TODAY - latest).days
    if age > FRESHNESS_LIMIT_DAYS:
        log_event(f"validate_{source}", "WARNING", "STALE_SOURCE",
                  f"Newest {source} row is {latest:%Y-%m-%d}, which is {age} days old (limit {FRESHNESS_LIMIT_DAYS})")
    else:
        log_event(f"validate_{source}", "INFO", "LATEST_DATE", f"Newest {source} row is {latest:%Y-%m-%d}")

In [8]:
ASX_REQUIRED = ["Open", "High", "Low", "Close", "Volume"]


# STEP 2 MAIN FUNCTIONS (TRANSFORM). Each reads a RAW file from disk, checks every row, and returns a clean table.


def transform_asx(path=None):
    """Turn the raw Yahoo file into a clean ASX 200 table with one row per trading date.

    Steps:
      1. Read the raw file from disk (as text, so nothing is changed) and check it has the expected columns.
      2. Convert the dates (with explicit time zone handling) and the numbers.
      3. Apply the row rules one at a time; rows that break a rule go to quarantine.
      4. Build the clean table and prove the row counts balance (raw = clean + quarantined).
      5. Add warnings about the data as a whole (repeated prices, date gaps, low volume, an open that repeats the previous close). Rows are kept for these.
    If path is not given, the newest raw ASX file is used.
    """
    path = path or latest_raw("asx200_yahoo")
    # --- 1. Read the file and check its layout ---
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)      # read everything as text so nothing is altered
    date_col = next((c for c in ("Date", "Datetime") if c in raw.columns), None)
    missing_cols = [c for c in ASX_REQUIRED if c not in raw.columns]
    if date_col is None or missing_cols:
        msg = f"Expected a date column plus {ASX_REQUIRED}; found {list(raw.columns)}"
        log_event("validate_asx", "CRITICAL", "SCHEMA_CHANGE", msg)
        raise SchemaError(msg)

    # Number each raw row so a rejected row can be traced back to its line in the file.
    # Columns whose names start with "_" are helper columns added here; they are not part of the original data.
    raw["_raw_row"] = np.arange(1, len(raw) + 1)
    n_raw = len(raw)
    log_event("validate_asx", "INFO", "ROWS_READ", f"Read {path.name}", count=n_raw)
    df = raw.copy()

    # --- 2. Convert dates and numbers ---
    # Explicit time zone handling: Yahoo gives midnight Sydney time with a UTC offset (+10:00 or +11:00).
    # Steps: parse with the offset -> convert to Sydney local time -> keep only the calendar date.
    stamp = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    df["_date"] = stamp.dt.tz_convert(LOCAL_TZ).dt.tz_localize(None).dt.normalize()

    # Convert each price/volume column from text to numbers; keep flags for blank and non-numeric cells.
    for col in ASX_REQUIRED:
        df[f"_{col.lower()}"], df[f"_miss_{col.lower()}"], df[f"_bad_{col.lower()}"] = parse_numeric(df[col])
    price_cols = ["_open", "_high", "_low", "_close"]
    any_missing = df[[f"_miss_{c}" for c in ("open", "high", "low", "close")]].any(axis=1)
    any_bad = df[[f"_bad_{c}" for c in ("open", "high", "low", "close", "volume")]].any(axis=1)

    # --- 3. Row rules. Each reject() removes the rows that break ONE rule; a row goes to quarantine on its first failure. ---
    df = reject("asx", df, df["_date"].isna(), "INVALID_DATE", "Date could not be read")
    df = reject("asx", df, df.duplicated("_date", keep="last"), "DUPLICATE_DATE",
                "Same trading date appears more than once; the last copy was kept")
    # If the run happens during the trading day, Yahoo returns today's unfinished bar. Reject it as incomplete.
    session_cutoff = pd.Timestamp(f"{TODAY:%Y-%m-%d} {ASX_CLOSE_CUTOFF}", tz=LOCAL_TZ)
    df = reject("asx", df, (df["_date"] == TODAY) & (NOW_LOCAL < session_cutoff), "INCOMPLETE_SESSION",
                f"Bar dated today was fetched before {ASX_CLOSE_CUTOFF} Sydney time, so the trading session may not have finished")
    df = reject("asx", df, any_missing.reindex(df.index), "MISSING_VALUE", "Open, high, low or close is blank")
    df = reject("asx", df, any_bad.reindex(df.index), "UNPARSEABLE_VALUE", "A value is present but is not a number")
    df = reject("asx", df, (df[price_cols] <= 0).any(axis=1), "NON_POSITIVE_PRICE", "A price is zero or negative")

    # Consistency check: high must be at least as large as open, close and low; low must be no larger than open, close and high.
    # A tiny tolerance (OHLC_TOLERANCE) avoids rejecting rows over rounding.
    tol = 1 + OHLC_TOLERANCE
    top = df[["_open", "_close", "_low"]].max(axis=1)
    bottom = df[["_open", "_close", "_high"]].min(axis=1)
    inconsistent = (df["_high"] * tol < top) | (df["_low"] > bottom * tol)
    df = reject("asx", df, inconsistent, "OHLC_INCONSISTENT", "High is below open/close/low, or low is above open/close/high")
    df = reject("asx", df, df["_volume"] < 0, "NEGATIVE_VOLUME", "Volume is below zero")
    # A real trading day always has a range. Open = high = low = close means a placeholder or unfinished bar.
    flat = (df["_open"] == df["_high"]) & (df["_high"] == df["_low"]) & (df["_low"] == df["_close"])
    df = reject("asx", df, flat, "ZERO_RANGE_BAR",
                "Open, high, low and close are identical (zero trading range)")

    # --- 4. Build the clean table from the rows that passed every rule, then prove no rows vanished ---
    clean = pd.DataFrame({
        "date": df["_date"], "open": df["_open"], "high": df["_high"],
        "low": df["_low"], "close": df["_close"], "volume": df["_volume"],
    }).sort_values("date").reset_index(drop=True)
    clean["volume"] = clean["volume"].round().astype("Int64")
    reconcile("asx", n_raw, len(clean))
    require_rows("asx", clean)

    # --- 5. Warnings about the data as a whole. Rows are KEPT; each warning is logged so the person using the data can see it. ---
    # Identical open, high, low and close on two consecutive trading dates.
    cols = ["open", "high", "low", "close"]
    stale = (clean[cols] == clean[cols].shift()).all(axis=1)
    if stale.any():
        log_event("validate_asx", "WARNING", "STALE_PRICES",
                  f"Identical OHLC on consecutive trading days: {sample_dates(clean.loc[stale, 'date'])}", count=int(stale.sum()))
    # More than MAX_CALENDAR_GAP_DAYS calendar days between consecutive trading dates.
    gap = clean["date"].diff().dt.days > MAX_CALENDAR_GAP_DAYS
    if gap.any():
        log_event("validate_asx", "WARNING", "CALENDAR_GAP",
                  f"More than {MAX_CALENDAR_GAP_DAYS} days since the previous row, ending: {sample_dates(clean.loc[gap, 'date'])}", count=int(gap.sum()))
    # Volume: warn if the median is zero or missing, or list the days with volume under LOW_VOLUME_FRACTION of the median.
    volumes = clean["volume"].astype("float64")
    median_volume = volumes.median()
    if not median_volume > 0:                              # volume is zero or missing on most days
        log_event("validate_asx", "WARNING", "ZERO_MEDIAN_VOLUME",
                  "Median volume is zero or missing.")
    else:
        low_vol = volumes < LOW_VOLUME_FRACTION * median_volume
        if low_vol.any():
            log_event("validate_asx", "WARNING", "LOW_VOLUME_OUTLIER",
                      f"Volume under {LOW_VOLUME_FRACTION:.0%} of the median ({median_volume:,.0f}) on these days: {sample_dates(clean.loc[low_vol, 'date'], 20)}",
                      count=int(low_vol.sum()))
    # Measure how often "open" equals the previous day's close, and warn if that share is above OPEN_MATCH_WARN_SHARE.
    previous_close = clean["close"].shift()
    same_as_prev = (clean["open"] - previous_close).abs() < 0.01
    open_share = float(same_as_prev.iloc[1:].mean()) if len(clean) > 1 else 0.0
    if open_share > OPEN_MATCH_WARN_SHARE:
        log_event("validate_asx", "WARNING", "OPEN_EQUALS_PREVIOUS_CLOSE",
                  f"'open' equals the previous day's close on {open_share:.0%} of days (warning threshold {OPEN_MATCH_WARN_SHARE:.0%}). On those days open-to-close change equals close-to-close change.",
                  count=int(same_as_prev.iloc[1:].sum()))
    # Record the date range, and warn if the newest row is old.
    log_event("validate_asx", "INFO", "DATE_RANGE", f"{clean['date'].min():%Y-%m-%d} to {clean['date'].max():%Y-%m-%d}", count=len(clean))
    check_freshness("asx", clean["date"].max())
    return clean

In [9]:
def transform_vix(path=None):
    """Turn the raw FRED file into a clean VIX table with one row per US market date.

    Same pattern as transform_asx, but simpler because the VIX is a single number per day: read the raw file, check its layout, convert dates and numbers, quarantine rows that break a rule, and prove the row counts balance. If path is not given, the newest raw VIX file is used.
    """
    path = path or latest_raw("vix_fred")
    # --- 1. Read the file and check its layout ---
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    date_col = next((c for c in ("observation_date", "DATE") if c in raw.columns), None)   # FRED has used both names
    if date_col is None or FRED_SERIES not in raw.columns:
        msg = f"Expected a date column and '{FRED_SERIES}'; found {list(raw.columns)}"
        log_event("validate_vix", "CRITICAL", "SCHEMA_CHANGE", msg)
        raise SchemaError(msg)

    # Number each raw row for traceability (helper columns start with "_").
    raw["_raw_row"] = np.arange(1, len(raw) + 1)
    n_raw = len(raw)
    log_event("validate_vix", "INFO", "ROWS_READ", f"Read {path.name} (date column: {date_col})", count=n_raw)
    df = raw.copy()

    # --- 2. Convert dates and numbers ---
    df["_date"] = pd.to_datetime(df[date_col], errors="coerce").dt.normalize()   # US market date, no time zone
    df["_vix"], df["_miss"], df["_bad"] = parse_numeric(df[FRED_SERIES])

    # --- 3. Row rules (same idea as transform_asx). A blank VIX value is rejected here, then carried forward in the align step. ---
    df = reject("vix", df, df["_date"].isna(), "INVALID_DATE", "Date could not be read")
    df = reject("vix", df, df.duplicated("_date", keep="last"), "DUPLICATE_DATE",
                "Same date appears more than once; the last copy was kept")
    df = reject("vix", df, df["_miss"], "MISSING_VALUE",
                "FRED has no VIX close for this date. If an ASX date would have used it, the align step uses the most recent earlier close and sets vix_is_filled to True.",
                level="INFO")
    df = reject("vix", df, df["_bad"], "UNPARSEABLE_VALUE", "A value is present but is not a number")
    df = reject("vix", df, df["_vix"] <= 0, "NON_POSITIVE_PRICE", "VIX value is zero or negative")

    # --- 4. Build the clean table and prove no rows vanished ---
    clean = pd.DataFrame({"vix_source_date": df["_date"], "vix_close": df["_vix"]}) \
        .sort_values("vix_source_date").reset_index(drop=True)
    reconcile("vix", n_raw, len(clean))
    require_rows("vix", clean)

    log_event("validate_vix", "INFO", "DATE_RANGE",
              f"{clean['vix_source_date'].min():%Y-%m-%d} to {clean['vix_source_date'].max():%Y-%m-%d}", count=len(clean))
    check_freshness("vix", clean["vix_source_date"].max())
    return clean

In [10]:
# RUN THE TRANSFORMS. Each reads its raw file from disk and returns a clean table.
asx_clean = transform_asx(ASX_RAW_PATH)
vix_clean = transform_vix(VIX_RAW_PATH)
print(f"\nASX clean rows: {len(asx_clean)} | VIX clean rows: {len(vix_clean)}")

2026-10-26 13:53:18,829 INFO     [validate_asx] ROWS_READ: Read asx200_yahoo_20261026T135317.csv (n=506)
2026-10-26 13:53:18,874 WARNING  [validate_asx] INCOMPLETE_SESSION: Bar dated today was fetched before 16:30 Sydney time, so the trading session may not have finished (n=1)
2026-10-26 13:53:18,887 WARNING  [validate_asx] ZERO_RANGE_BAR: Open, high, low and close are identical (zero trading range) (n=1)
2026-10-26 13:53:18,891 INFO     [reconcile_asx] ROW_RECONCILIATION: raw=506, clean=504, quarantined=2, balanced=True
2026-10-26 13:53:18,898 WARNING  [validate_asx] LOW_VOLUME_OUTLIER: Volume under 20% of the median (683,200) on these days: 2024-11-07, 2025-01-06, 2025-01-20, 2025-01-21, 2025-01-30, 2025-02-17, 2025-11-06, 2025-11-10, 2025-11-27, 2026-01-06, 2026-01-29, 2026-02-09, 2026-02-16, 2026-03-02, 2026-03-25 (n=15)
2026-10-26 13:53:18,901 WARNING  [validate_asx] OPEN_EQUALS_PREVIOUS_CLOSE: 'open' equals the previous day's close on 91% of days (warning threshold 50%). On those


ASX clean rows: 504 | VIX clean rows: 523


## Step 3. Align the two sources on one time axis

The ASX and the US market trade in different time zones. The ASX session for a Sydney date *D* runs 10:00 to 16:00 Sydney time. The US close for date *D* happens at 4pm New York time, which is the morning of D+1 in Sydney, after the ASX has already closed for D. So the VIX close for the same calendar date is not yet known while the ASX is trading.

**The rule.**

> For each ASX date *D*, use the most recent VIX close from a date strictly before *D*.

* The ASX 200 is the date spine (source of truth): the output has exactly one row per clean ASX trading date.
* On a normal Tuesday to Friday this is the previous US trading day's close. On a Monday it is Friday's close.
* Using the same-date VIX would be look-ahead bias (it would use information that did not exist yet), so exact date matches are not allowed.
* This holds all year round. Daylight saving moves the gap between the two markets by an hour or two, but never changes which US close happens first.

**Filled rows.** If the expected VIX date has no value (for example Christmas Day or New Year's Day, or because FRED has not published it yet), the previous available close is used and the row is flagged `vix_is_filled = True`. The expected date is the previous weekday. The flag, plus `vix_source_date`, ensures the user can see exactly which VIX date each row used.

**Safety limit.** If no VIX close exists within 7 days, the VIX cell is left blank and logged.

In [11]:
def align(asx, vix):
    """Join the two clean tables into one, using the ASX dates as the backbone ("date spine").

    Rule: for each ASX date, use the most recent VIX close from a date STRICTLY BEFORE it. The US market closes after the ASX has closed for the same date, so the same-date VIX is not yet knownwhile the ASX is trading; using it would be look-ahead bias. A Monday gets Friday's VIX, a Tuesday gets Monday's.

    The result has the same number of rows as the ASX table, plus three columns:
      vix_close        the VIX value used
      vix_source_date  the US date that value came from
      vix_is_filled    True when the usual previous-weekday value was missing, so an older value was carried forward
    """
    asx = asx.sort_values("date").reset_index(drop=True)
    vix = vix.sort_values("vix_source_date").reset_index(drop=True)
    asx["date"] = asx["date"].astype("datetime64[ns]")                       # keys must share one dtype
    vix["vix_source_date"] = vix["vix_source_date"].astype("datetime64[ns]")

    # merge_asof matches each ASX date to the nearest VIX date at or before it. allow_exact_matches=False makes it
    # strictly before. tolerance stops a VIX value being carried forward more than 7 days (it is left blank instead).
    out = pd.merge_asof(
        asx, vix,
        left_on="date", right_on="vix_source_date",
        direction="backward",
        allow_exact_matches=False,                                           # strictly earlier: no look-ahead
        tolerance=pd.Timedelta(days=VIX_MAX_STALENESS_DAYS),
    )

    # Flag carried-forward values. The VIX date we normally expect is the previous weekday (Friday for a Monday).
    # If the value used is older than that, the expected date had no data, so the value was carried forward.
    previous_weekday = pd.Series(
        pd.to_datetime(np.busday_offset(out["date"].values.astype("datetime64[D]"), -1, roll="backward")).values.astype("datetime64[ns]"),
        index=out.index,
    )
    out["vix_is_filled"] = out["vix_source_date"].notna() & (out["vix_source_date"] < previous_weekday)

    # Log what the join did: the rule used, which rows were carried forward, any rows left without a VIX value.
    log_event("align", "INFO", "ALIGNMENT_RULE",
              "Each ASX date uses the latest VIX close from a strictly earlier date (ASX 200 is the date spine)", count=len(out))
    filled = out["vix_is_filled"]
    if filled.any():
        log_event("align", "INFO", "VIX_CARRIED_FORWARD",
                  f"Previous VIX close used because the expected date had no value: {sample_dates(out.loc[filled, 'date'])}", count=int(filled.sum()))
    no_vix = out["vix_close"].isna()
    if no_vix.any():
        log_event("align", "WARNING", "NO_VIX_WITHIN_TOLERANCE",
                  f"No VIX close within {VIX_MAX_STALENESS_DAYS} days before: {sample_dates(out.loc[no_vix, 'date'])}", count=int(no_vix.sum()))
    max_lag = int((out["date"] - out["vix_source_date"]).dt.days.max())
    log_event("align", "INFO", "VIX_MAX_LAG_DAYS", f"Largest gap between an ASX date and its VIX date: {max_lag} days")
    return out


# RUN THE ALIGNMENT.
curated = align(asx_clean, vix_clean)

2026-10-26 13:53:18,955 INFO     [align] ALIGNMENT_RULE: Each ASX date uses the latest VIX close from a strictly earlier date (ASX 200 is the date spine) (n=504)
2026-10-26 13:53:18,957 INFO     [align] VIX_CARRIED_FORWARD: Previous VIX close used because the expected date had no value: 2025-01-02, 2026-01-02 (n=2)
2026-10-26 13:53:18,960 INFO     [align] VIX_MAX_LAG_DAYS: Largest gap between an ASX date and its VIX date: 3 days


## Step 4. Final integrity checks and the curated file

Before anything is written, four checks must pass. Any failure stops the run, so a wrong file is never handed over.

1. One output row per clean ASX date (nothing added or lost in the join).
2. Dates are unique and in order.
3. No missing open, high, low or close.
4. No look-ahead: every VIX source date is strictly earlier than its ASX date.

The output file is `data/curated/market_curated.csv`.

In [12]:
def final_checks(out, asx):
    """Last safety net before anything is written. Stops the run if the combined table has a problem.

    Checks:
    (1) the join did not add or lose rows,
    (2) dates are unique and in order,
    (3) no open/high/low/close is missing,
    (4) no VIX date is on or after its ASX date (no look-ahead).

    If any check fails, each problem is logged as CRITICAL and the run stops, so a wrong file is never handed over.
    """
    problems = []
    if len(out) != len(asx):
        problems.append(f"row count changed in join: {len(asx)} -> {len(out)}")
    if not out["date"].is_unique or not out["date"].is_monotonic_increasing:
        problems.append("dates are not unique and sorted")
    if out[["open", "high", "low", "close"]].isna().any().any():
        problems.append("missing OHLC values in output")
    if (out["vix_source_date"].dropna() >= out.loc[out["vix_source_date"].notna(), "date"]).any():
        problems.append("look-ahead: a VIX date is not strictly earlier than its ASX date")
    if problems:
        for p in problems:
            log_event("final_checks", "CRITICAL", "INTEGRITY_CHECK_FAILED", p)
        raise PipelineError("; ".join(problems))
    log_event("final_checks", "INFO", "INTEGRITY_CHECKS_PASSED", "Row count, ordering, OHLC completeness and no-look-ahead checks passed", count=len(out))


# RUN THE CHECKS, THEN WRITE THE FINAL FILE. This is the single CSV handed on for metric building.
final_checks(curated, asx_clean)

OUTPUT_COLUMNS = ["date", "open", "high", "low", "close", "volume", "vix_close", "vix_source_date", "vix_is_filled"]
curated[OUTPUT_COLUMNS].to_csv(CURATED_FILE, index=False, date_format="%Y-%m-%d")
log_event("write", "INFO", "CURATED_WRITTEN", f"Wrote {CURATED_FILE}", count=len(curated))
curated[OUTPUT_COLUMNS].tail(8)

2026-10-26 13:53:18,974 INFO     [final_checks] INTEGRITY_CHECKS_PASSED: Row count, ordering, OHLC completeness and no-look-ahead checks passed (n=504)
2026-10-26 13:53:18,986 INFO     [write] CURATED_WRITTEN: Wrote data/curated/market_curated.csv (n=504)


,date,open,high,low,close,volume,vix_close,vix_source_date,vix_is_filled
496,2026-10-14,8783.900391,8783.900391,8698.299805,8727.900391,726000,15.27,2026-10-13,False
497,2026-10-15,8727.900391,8735.900391,8641.099609,8719.700195,726000,16.87,2026-10-14,False
498,2026-10-16,8719.700195,8824.500000,8719.700195,8806.700195,1013000,16.46,2026-10-15,False
499,2026-10-19,8806.700195,8849.900391,8800.900391,8845.700195,826900,19.99,2026-10-16,False
500,2026-10-20,8845.700195,8845.700195,8747.299805,8816.900391,798300,18.44,2026-10-19,False
501,2026-10-21,8816.900391,8841.900391,8795.900391,8831.299805,919200,17.31,2026-10-20,False
502,2026-10-22,8831.299805,8831.299805,8764.000000,8793.200195,814800,16.68,2026-10-21,False
503,2026-10-23,8793.200195,8795.200195,8752.599609,8795.200195,970700,19.03,2026-10-22,False


## Step 5. Data dictionary and run summary

The data dictionary is saved next to the curated file so the person building the metrics knows exactly what each column means.

In [13]:
# The data dictionary explains each column of market_curated.csv. It is saved next to the data file so the person building metrics sees what each column means and which checks apply to it.
data_dictionary = pd.DataFrame([
    ("date", "date (YYYY-MM-DD)", "ASX trading date in Sydney time (Australia/Sydney). One row per date. The other columns are matched to this date.", "Yahoo Finance"),
    ("open", "number", f"Opening level as reported by Yahoo Finance. Yahoo can report the previous day's close in place of the opening level. Each run measures how often this happens and logs OPEN_EQUALS_PREVIOUS_CLOSE when it is more than {OPEN_MATCH_WARN_SHARE:.0%} of days.", "Yahoo Finance"),
    ("high", "number", "Highest level during the session, as reported by Yahoo Finance. Rows where high is below open, close or low are quarantined (OHLC_INCONSISTENT).", "Yahoo Finance"),
    ("low", "number", "Lowest level during the session, as reported by Yahoo Finance. Rows where low is above open, close or high are quarantined (OHLC_INCONSISTENT).", "Yahoo Finance"),
    ("close", "number", "Closing level for the session, as reported by Yahoo Finance. Not dividend-adjusted.", "Yahoo Finance"),
    ("volume", "integer", f"Volume as reported by Yahoo Finance. The source does not state the unit. Days with volume under {LOW_VOLUME_FRACTION:.0%} of the median are listed in the log (LOW_VOLUME_OUTLIER) and kept in the data.", "Yahoo Finance"),
    ("vix_close", "number", f"VIX (CBOE Volatility Index) close from FRED series {FRED_SERIES}, taken from the latest US market date strictly before `date`. Blank if no value exists within {VIX_MAX_STALENESS_DAYS} days.", "FRED"),
    ("vix_source_date", "date (YYYY-MM-DD)", "The US market date that `vix_close` was taken from. Always earlier than `date`.", "FRED"),
    ("vix_is_filled", "True/False", "True when no VIX value existed for the previous weekday before `date`, so the most recent earlier value was used. Otherwise False.", "Derived"),
], columns=["column", "type", "description", "source"])

data_dictionary.to_csv(CURATED_DIR / "data_dictionary.csv", index=False)
log_event("write", "INFO", "DATA_DICTIONARY_WRITTEN", f"Wrote {CURATED_DIR / 'data_dictionary.csv'}", count=len(data_dictionary))
pd.set_option("display.max_colwidth", 120)
data_dictionary

2026-10-26 13:53:19,029 INFO     [write] DATA_DICTIONARY_WRITTEN: Wrote data/curated/data_dictionary.csv (n=9)


,column,type,description,source
0,date,date (YYYY-MM-DD),ASX trading date in Sydney time (Australia/Sydney). One row per date. The other columns are matched to this date.,Yahoo Finance
1,open,number,Opening level as reported by Yahoo Finance. Yahoo can report the previous day's close in place of the opening level....,Yahoo Finance
2,high,number,"Highest level during the session, as reported by Yahoo Finance. Rows where high is below open, close or low are quar...",Yahoo Finance
3,low,number,"Lowest level during the session, as reported by Yahoo Finance. Rows where low is above open, close or high are quara...",Yahoo Finance
4,close,number,"Closing level for the session, as reported by Yahoo Finance. Not dividend-adjusted.",Yahoo Finance
5,volume,integer,Volume as reported by Yahoo Finance. The source does not state the unit. Days with volume under 20% of the median ar...,Yahoo Finance
6,vix_close,number,"VIX (CBOE Volatility Index) close from FRED series VIXCLS, taken from the latest US market date strictly before `dat...",FRED
7,vix_source_date,date (YYYY-MM-DD),The US market date that `vix_close` was taken from. Always earlier than `date`.,FRED
8,vix_is_filled,True/False,"True when no VIX value existed for the previous weekday before `date`, so the most recent earlier value was used. Ot...",Derived


In [14]:
# ---- Run summary: what came in, what was set aside, and what went out ----
summary = pd.DataFrame(COUNTS).T
summary["curated_rows"] = [len(curated) if src == "asx" else "" for src in summary.index]
print("ROW RECONCILIATION (raw = clean + quarantined)")
display(summary)

events = pd.DataFrame(EVENTS)
print("\nEVENTS BY LEVEL AND CODE")
display(events.groupby(["level", "code"]).size().rename("events").reset_index())

quarantined = pd.read_csv(QUARANTINE_LOG)
print(f"\nQUARANTINED ROWS: {len(quarantined)} (details in {QUARANTINE_LOG.name})")
if len(quarantined):
    display(quarantined.groupby(["source", "reason_code"]).size().rename("rows").reset_index())

print("\nWARNINGS AND ABOVE")
display(events.loc[events["level"].isin(["WARNING", "ERROR", "CRITICAL"]), ["step", "level", "code", "message", "count"]])

print("\nFILES WRITTEN")
for p in (ASX_RAW_PATH, VIX_RAW_PATH, CURATED_FILE, CURATED_DIR / "data_dictionary.csv", EVENT_LOG, QUARANTINE_LOG, RUN_LOG):
    print(" ", p)

ROW RECONCILIATION (raw = clean + quarantined)


,raw_rows,clean_rows,quarantined_rows,balanced,curated_rows
asx,506,504,2,True,504
vix,529,523,6,True,



EVENTS BY LEVEL AND CODE


,level,code,events
0,INFO,ALIGNMENT_RULE,1
1,INFO,CURATED_WRITTEN,1
2,INFO,DATA_DICTIONARY_WRITTEN,1
3,INFO,DATE_RANGE,2
4,INFO,FETCH_OK,2
5,INFO,INTEGRITY_CHECKS_PASSED,1
6,INFO,LATEST_DATE,2
7,INFO,MISSING_VALUE,1
8,INFO,ROWS_READ,2
9,INFO,ROW_RECONCILIATION,2



QUARANTINED ROWS: 8 (details in quarantine_20261026T135317.csv)


,source,reason_code,rows
0,asx,INCOMPLETE_SESSION,1
1,asx,ZERO_RANGE_BAR,1
2,vix,MISSING_VALUE,6



WARNINGS AND ABOVE


,step,level,code,message,count
4,validate_asx,WARNING,INCOMPLETE_SESSION,"Bar dated today was fetched before 16:30 Sydney time, so the trading session may not have finished",1.0
5,validate_asx,WARNING,ZERO_RANGE_BAR,"Open, high, low and close are identical (zero trading range)",1.0
7,validate_asx,WARNING,LOW_VOLUME_OUTLIER,"Volume under 20% of the median (683,200) on these days: 2024-11-07, 2025-01-06, 2025-01-20, 2025-01-21, 2025-01-30, ...",15.0
8,validate_asx,WARNING,OPEN_EQUALS_PREVIOUS_CLOSE,'open' equals the previous day's close on 91% of days (warning threshold 50%). On those days open-to-close change eq...,460.0



FILES WRITTEN
  data/raw/asx200_yahoo_20261026T135317.csv
  data/raw/vix_fred_20261026T135317.csv
  data/curated/market_curated.csv
  data/curated/data_dictionary.csv
  logs/quality_log_20261026T135317.csv
  logs/quarantine_20261026T135317.csv
  logs/run_20261026T135317.log


## Step 6. Download the outputs

Colab's disk is wiped when the runtime resets, so files need to be downloaded. This bundles `data/` and `logs/` into one zip. If the download does not appear, open the folder icon in Colab's left sidebar, go to `data/curated/`, and right-click `market_curated.csv` to download it directly.

In [15]:
import zipfile   # builds the .zip file

# Put everything (data/ and logs/) into one zip so a single download captures the whole run.
archive = Path(f"pipeline_run_{RUN_ID}.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (ROOT / "data", ROOT / "logs"):
        for f in sorted(folder.rglob("*")):
            if f.is_file():
                z.write(f)
print("Created", archive, f"({archive.stat().st_size / 1024:.0f} KB) containing data/ and logs/")

try:
    from google.colab import files
    files.download(str(archive))          # one download only: browsers often block a second automatic download
except ImportError:
    print("Not running in Colab; collect the zip file from the working folder.")

Created pipeline_run_20261026T135317.zip (28 KB) containing data/ and logs/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>